## PATSTAT focus on the applications table - lesson 2
This notebook expands on the first lesson about PATSTAT. We will take a look at the applications table, which is the main table of the database schema of Patstat. 

## The applications table
When working with PATSTAT you should be familiar with its rich data structure. The goal of this course is not to explain the whole set of tables and fields, since this can be found in the official documentation. We will, however, take a look at some of the main tables and work with them to perform data analysis. 

Table `tls201_appln`, that we will be refering to as the `applications table` is the central table in the databse schema for PATSTAT Global. Almost all other tables in the schema have a direct relationship with the applications table. Let's take a look at this table


### Viewing all the fields with SQL
We saw in lesson 1 that we can query PATSTAT using SQL language, using the  method `sql_query`. We are going to take advantage of that and query using `SELECT * FROM tls201_appln`, which gives us all the fields in a given table.  

In [1]:
# Importing the PATSTAT client
from epo.tipdata.patstat import PatstatClient

# Initialise the PATSTAT client
patstat = PatstatClient()

# Access ORM
db = patstat.orm()


This client instance is currently configured to use a test dataset with reduced number of publications (~10K).
Use PatstatClient(env='PROD') to use the complete PATSTAT dataset (>140M publications).
Use PatstatClient(env='TEST') to use the test dataset and avoid displaying this warning



In [2]:
# Querying with SQL for all the fields in the applications table

res = patstat.sql_query("""
SELECT
    *
FROM
    tls201_appln
""")

# Printing the number of fields
print (f"Number of fields in the applications table", len(res[0]))

# Showing the first result 
res[0]

Number of fields in the applications table 27


{'appln_id': 23307078,
 'appln_auth': 'GB',
 'appln_nr': '1020555D',
 'appln_kind': 'A ',
 'appln_filing_date': datetime.date(9999, 12, 31),
 'appln_filing_year': 9999,
 'appln_nr_original': None,
 'ipr_type': 'PI',
 'receiving_office': '  ',
 'internat_appln_id': 0,
 'int_phase': 'N',
 'reg_phase': 'N',
 'nat_phase': 'Y',
 'earliest_filing_date': datetime.date(9999, 12, 31),
 'earliest_filing_year': 9999,
 'earliest_filing_id': 23307078,
 'earliest_publn_date': datetime.date(9999, 12, 31),
 'earliest_publn_year': 9999,
 'earliest_pat_publn_id': 321362796,
 'granted': 'N',
 'docdb_family_id': 1755039,
 'inpadoc_family_id': 23307078,
 'docdb_family_size': 1,
 'nb_citing_docdb_fam': 5,
 'nb_applicants': 0,
 'nb_inventors': 0,
 'appln_nr_epodoc': None}

### The fields in the applications table
We can see that table `tls201_appln` in the PATSTAT database contains 27 fields. Below you can see a description of each field. 


1. **appln_id**: a unique identifier for the patent application. This is an internal number used to uniquely identify each application within the database.

2. **appln_auth**: the patent authority or office where the application was filed. For example, 'EP' stands for the European Patent Office.

3. **appln_nr**: the application number assigned by the patent authority. This number is unique within the context of the authority.

4. **appln_kind**: the kind of patent document, often represented by a letter (e.g., 'A' for a published application, 'B' for a granted patent).

5. **appln_filing_date**: the date on which the patent application was filed with the patent authority.

6. **appln_filing_year**: the year in which the patent application was filed, extracted from the filing date.

7. **appln_nr_epodoc**: the application number in the EPODOC format, a standardized format used by the European Patent Office.

8. **appln_nr_original**: the original application number as assigned by the patent authority.

9. **ipr_type**: the type of intellectual property right, such as 'PI' for patent of invention.

10. **receiving_office**: the receiving office for the application, which is typically used in the context of international patent applications.

11. **internat_appln_id**: identifier for the international application, if applicable.

12. **int_phase**: indicates whether the application is in the international phase ('Y' for yes, 'N' for no).

13. **reg_phase**: indicates whether the application is in the regional phase ('Y' for yes, 'N' for no).

14. **nat_phase**: indicates whether the application is in the national phase ('Y' for yes, 'N' for no).

15. **earliest_filing_date**: the earliest filing date among all related applications in the same patent family.

16. **earliest_filing_year**: the year of the earliest filing date.

17. **earliest_filing_id**: identifier for the earliest related application.

18. **earliest_publn_date**: the earliest publication date of the application.

19. **earliest_publn_year**: the year of the earliest publication date.

20. **earliest_pat_publn_id**: identifier for the earliest related publication.

21. **granted**: indicates whether the application has been granted ('Y' for yes, 'N' for no).

22. **docdb_family_id**: identifier for the DOCDB patent family, which groups related patent documents across different countries.

23. **inpadoc_family_id**: identifier for the INPADOC patent family, a broader grouping of related patent documents.

24. **docdb_family_size**: the number of documents in the DOCDB patent family.

25. **nb_citing_docdb_fam**: the number of DOCDB patent families that cite this application.

26. **nb_applicants**: the number of applicants for the patent.

27. **nb_inventors**: the number of inventors listed on the application.

## Example query: the most influencial European patents of the decade
Before moving to more complex queries, let's take a look at an example of a query using only the applications table.

We will use ORM for this example and throughout the rest of the course. Remember that for working with ORM, we need to import the table(s) we want to work with as models. 

In [3]:
# Importing tables as models
from epo.tipdata.patstat.database.models import TLS201_APPLN

### Our query
We will see what granted European patents have been cited the most, from the applications filed in this decade. 

In [4]:
# Importing necessary modules
from epo.tipdata.patstat.database.models import TLS201_APPLN

# Define the query in ORM
q = db.query(
    TLS201_APPLN.appln_id,
    TLS201_APPLN.appln_nr,
    TLS201_APPLN.appln_filing_date,
    TLS201_APPLN.nb_citing_docdb_fam  # number of families citing the application
).filter(
    TLS201_APPLN.appln_filing_year >= 2020,
    TLS201_APPLN.appln_auth == 'EP',
    TLS201_APPLN.granted == 'Y'
).order_by(
    TLS201_APPLN.nb_citing_docdb_fam.desc()
)

# Creating a dataframe with the results
res= patstat.df(q)

res


,appln_id,appln_nr,appln_filing_date,nb_citing_docdb_fam
0,545001881,20845652,2020-08-31,12
1,550137442,21172558,2021-05-06,12
2,550137444,21172559,2021-05-06,12
3,532706979,20179844,2020-06-12,11
4,544678369,21153218,2021-01-25,11
...,...,...,...,...
533,524931428,20151578,2020-01-14,0
534,578613921,22192729,2022-08-30,0
535,568000314,22705015,2022-02-08,0
536,573429931,22725234,2022-04-25,0


### Adding a link to the EP Register
We will now add a link to the European Patent Register for the top 10 most cited granted patents. We use the application number of each record, and generate the url for the register with that application number. 

In [5]:
# Extract the first 10 records
top_10_records = res.head(10)

# Loop over the first 10 records and generate the URLs
urls = []
for index, row in top_10_records.iterrows():
    appln_nr = row['appln_nr']
    print (f"https://register.epo.org/application?number=EP{appln_nr}")
   

https://register.epo.org/application?number=EP20845652
https://register.epo.org/application?number=EP21172558
https://register.epo.org/application?number=EP21172559
https://register.epo.org/application?number=EP20179844
https://register.epo.org/application?number=EP21153218
https://register.epo.org/application?number=EP20162363
https://register.epo.org/application?number=EP20715065
https://register.epo.org/application?number=EP20785109
https://register.epo.org/application?number=EP20192912
https://register.epo.org/application?number=EP20173683
